# geoutils smoke test

Smoke test for `geoutils`, which wraps rasterio and geopandas in `Raster` and `Vector` classes.
Every example builds its data in memory on a small 10 m grid in UTM zone 31N (EPSG:32631);
nothing is downloaded.

## Example 1: A Raster from a NumPy array

In [ ]:
import geoutils as gu
import numpy as np
import rasterio.transform

# Example 1: a Raster built from a NumPy array, a transform and a CRS.
# 10 x 10 pixels of 10 m in UTM zone 31N (EPSG:32631); the top-left pixel is marked as nodata.
values = np.arange(100, dtype=np.float32).reshape(10, 10)
mask = np.zeros_like(values, dtype=bool)
mask[0, 0] = True
transform = rasterio.transform.from_origin(500000, 4500100, 10, 10)
r = gu.Raster.from_array(np.ma.masked_array(values, mask=mask), transform=transform, crs=32631, nodata=-9999)

print(f"shape={r.shape} res={r.res} crs=EPSG:{r.crs.to_epsg()}")
print(f"bounds={tuple(r.bounds)}")
assert r.shape == (10, 10) and r.res == (10.0, 10.0)
assert r.crs.to_epsg() == 32631 and r.nodata == -9999
assert tuple(r.bounds) == (500000.0, 4500000.0, 500100.0, 4500100.0)
assert r.data.mask[0, 0] and r.data.count() == 99  # the nodata pixel is masked

# Arithmetic returns a new Raster on the same grid and keeps the mask.
doubled = r * 2 + 1
assert isinstance(doubled, gu.Raster) and doubled.shape == r.shape
assert doubled.data.mask[0, 0] and float(doubled.data[1, 1]) == 2 * values[1, 1] + 1
print(f"valid mean={float(r.data.mean()):.2f} | doubled+1 max={float(doubled.data.max())}")

## Example 2: A Vector from a GeoDataFrame, and reprojection

In [ ]:
import geoutils as gu
import geopandas as gpd
from shapely.geometry import box

# Example 2: a Vector wrapping a GeoDataFrame, and reprojection to geographic coordinates.
field = gpd.GeoDataFrame({"name": ["field"]}, geometry=[box(500020, 4500020, 500050, 4500060)], crs=32631)
v = gu.Vector(field)
assert isinstance(v.ds, gpd.GeoDataFrame) and v.ds["name"].tolist() == ["field"]
assert v.crs.to_epsg() == 32631
assert tuple(v.bounds) == (500020.0, 4500020.0, 500050.0, 4500060.0)
assert abs(float(v.ds.area.iloc[0]) - 30 * 40) < 1e-6  # 30 m x 40 m
print(f"vector bounds={tuple(float(b) for b in v.bounds)} area={float(v.ds.area.iloc[0])} m^2")

# UTM zone 31N is centred on 3 degrees east; this box sits at about 40.6 degrees north.
lonlat = v.reproject(crs=4326)
assert isinstance(lonlat, gu.Vector) and lonlat.crs.to_epsg() == 4326
left, bottom, right, top = (float(b) for b in lonlat.bounds)
assert 2.99 < left < right < 3.01 and 40.0 < bottom < top < 41.0, (left, bottom, right, top)
print(f"in EPSG:4326: lon {left:.5f}..{right:.5f}, lat {bottom:.5f}..{top:.5f}")

## Example 3: Crop a Raster to a Vector

In [ ]:
import geoutils as gu
import geopandas as gpd
import numpy as np
import rasterio.transform
from shapely.geometry import box

# Example 3: crop a Raster to a Vector's extent.
values = np.arange(100, dtype=np.float32).reshape(10, 10)
r = gu.Raster.from_array(values, transform=rasterio.transform.from_origin(500000, 4500100, 10, 10),
                         crs=32631, nodata=-9999)
v = gu.Vector(gpd.GeoDataFrame(geometry=[box(500020, 4500020, 500050, 4500060)], crs=32631))

cropped = r.crop(v)
print(f"cropped shape={cropped.shape} bounds={tuple(cropped.bounds)}")
assert cropped.shape == (4, 3)
assert tuple(cropped.bounds) == (500020.0, 4500020.0, 500050.0, 4500060.0)
# Rows 4-7 and columns 2-4 of the source array cover that box.
assert np.array_equal(np.asarray(cropped.data), values[4:8, 2:5])
assert r.shape == (10, 10)  # crop returns a new Raster; the source is unchanged

## Example 4: Masks and rasterization

In [ ]:
import geoutils as gu
import geopandas as gpd
import numpy as np
import rasterio.transform
from shapely.geometry import box

# Example 4: boolean Masks from a comparison and from a Vector, and rasterization.
values = np.arange(100, dtype=np.float32).reshape(10, 10)
r = gu.Raster.from_array(values, transform=rasterio.transform.from_origin(500000, 4500100, 10, 10),
                         crs=32631, nodata=-9999)
v = gu.Vector(gpd.GeoDataFrame(geometry=[box(500020, 4500020, 500050, 4500060)], crs=32631))

high = r > 49.5
assert isinstance(high, gu.Raster) and high.is_mask
assert int(high.data.sum()) == 50 and high.shape == r.shape
assert float(r[high].min()) == 50.0  # indexing a Raster with a Mask selects those pixels

inside = v.create_mask(r)  # pixels covered by the polygon, on the raster's grid
assert inside.is_mask and int(inside.data.sum()) == 12  # 4 rows x 3 columns

burned = v.rasterize(r)  # 1 inside the polygon, 0 elsewhere
assert set(np.unique(np.asarray(burned.data)).tolist()) == {0, 1}
assert int(np.asarray(burned.data).sum()) == 12
both = int((high & inside).data.sum())
print(f"high pixels={int(high.data.sum())} | inside polygon={int(inside.data.sum())} | both={both}")
# The polygon covers rows 4-7, columns 2-4: row 4 (42-44) is below 49.5, rows 5-7 are above it.
assert both == int((values[4:8, 2:5] > 49.5).sum()) == 9

## Example 5: Resampling and reprojecting a Raster

In [ ]:
import geoutils as gu
import numpy as np
import rasterio.transform

# Example 5: resample a Raster to a finer grid and reproject it to another CRS.
values = np.arange(100, dtype=np.float32).reshape(10, 10)
r = gu.Raster.from_array(values, transform=rasterio.transform.from_origin(500000, 4500100, 10, 10),
                         crs=32631, nodata=-9999)

fine = r.reproject(res=5)  # same CRS and extent, half the pixel size
print(f"resampled: shape={fine.shape} res={fine.res}")
assert fine.shape == (20, 20) and fine.res == (5.0, 5.0)
assert tuple(fine.bounds) == tuple(r.bounds)

lonlat = r.reproject(crs=4326)
assert lonlat.crs.to_epsg() == 4326
left, bottom, right, top = lonlat.bounds
print(f"reprojected to EPSG:4326: lon {left:.5f}..{right:.5f}, lat {bottom:.5f}..{top:.5f}")
assert 2.99 < left < right < 3.01 and 40.0 < bottom < top < 41.0
assert np.isfinite(float(lonlat.data.mean()))